# 13. Vision transformer and detection blocks — complete small computation graphs

The spatial size is reduced, but the important paths are kept:

- ViT patch embedding + CLS
- Swin W-MSA **and** SW-MSA, including relative-position bias and shifted-window mask
- patch merging
- FPN lateral/top-down/smoothing path
- CenterNet-style prediction heads and box decoding


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. ViT patch sequence


In [ ]:
image = torch.randn(2, 3, 16, 16, device=device)
patch_size = 4
model_dim = 24

patch_projection = nn.Conv2d(
    3,
    model_dim,
    kernel_size=patch_size,
    stride=patch_size,
).to(device)

patch_tokens = patch_projection(image).flatten(2).transpose(1, 2)
cls_token = nn.Parameter(torch.zeros(1, 1, model_dim, device=device))
position = nn.Parameter(
    torch.randn(1, patch_tokens.size(1) + 1, model_dim, device=device) * 0.02
)

vit_tokens = torch.cat(
    [cls_token.expand(image.size(0), -1, -1), patch_tokens],
    dim=1,
)
vit_tokens = vit_tokens + position
print("ViT tokens:", vit_tokens.shape)


## 2. Swin window helpers and relative-position bias


In [ ]:
def window_partition(x, window_size):
    batch, height, width, channels = x.shape
    x = x.view(
        batch,
        height // window_size,
        window_size,
        width // window_size,
        window_size,
        channels,
    )
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    return x.view(-1, window_size * window_size, channels)


def window_reverse(windows, window_size, height, width, batch):
    channels = windows.size(-1)
    x = windows.view(
        batch,
        height // window_size,
        width // window_size,
        window_size,
        window_size,
        channels,
    )
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    return x.view(batch, height, width, channels)


def relative_position_index(window_size, device):
    coordinates = torch.stack(
        torch.meshgrid(
            torch.arange(window_size, device=device),
            torch.arange(window_size, device=device),
            indexing="ij",
        )
    )
    coordinates = coordinates.flatten(1)

    relative = coordinates[:, :, None] - coordinates[:, None, :]
    relative = relative.permute(1, 2, 0).contiguous()

    relative[:, :, 0] += window_size - 1
    relative[:, :, 1] += window_size - 1
    relative[:, :, 0] *= 2 * window_size - 1
    return relative.sum(dim=-1)


## 3. Shifted-window mask

Cyclic shift alone is not sufficient. Tokens that become adjacent only because
of wrap-around must be masked before attention.


In [ ]:
def build_shifted_window_mask(
    height,
    width,
    window_size,
    shift_size,
    device,
):
    region = torch.zeros(1, height, width, 1, device=device)

    h_slices = (
        slice(0, -window_size),
        slice(-window_size, -shift_size),
        slice(-shift_size, None),
    )
    w_slices = (
        slice(0, -window_size),
        slice(-window_size, -shift_size),
        slice(-shift_size, None),
    )

    region_id = 0
    for h_slice in h_slices:
        for w_slice in w_slices:
            region[:, h_slice, w_slice, :] = region_id
            region_id += 1

    mask_windows = window_partition(region, window_size).squeeze(-1)
    difference = mask_windows[:, None, :] - mask_windows[:, :, None]
    return difference == 0


## 4. Complete W-MSA / SW-MSA block


In [ ]:
class SwinWindowAttention(nn.Module):
    def __init__(
        self,
        model_dim=24,
        heads=3,
        window_size=2,
        shift_size=0,
    ):
        super().__init__()
        self.model_dim = model_dim
        self.heads = heads
        self.head_dim = model_dim // heads
        self.window_size = window_size
        self.shift_size = shift_size

        self.norm = nn.LayerNorm(model_dim)
        self.qkv = nn.Linear(model_dim, 3 * model_dim, bias=True)
        self.out = nn.Linear(model_dim, model_dim)

        bias_entries = (2 * window_size - 1) ** 2
        self.relative_bias = nn.Parameter(
            torch.zeros(bias_entries, heads)
        )

    def forward(self, x):
        batch, height, width, channels = x.shape
        residual = x
        x = self.norm(x)

        if self.shift_size > 0:
            x = torch.roll(
                x,
                shifts=(-self.shift_size, -self.shift_size),
                dims=(1, 2),
            )

        windows = window_partition(x, self.window_size)
        tokens_per_window = windows.size(1)

        qkv = self.qkv(windows).view(
            windows.size(0),
            tokens_per_window,
            3,
            self.heads,
            self.head_dim,
        ).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        index = relative_position_index(
            self.window_size,
            x.device,
        )
        bias = self.relative_bias[index.reshape(-1)]
        bias = bias.view(
            tokens_per_window,
            tokens_per_window,
            self.heads,
        ).permute(2, 0, 1)
        scores = scores + bias[None]

        if self.shift_size > 0:
            base_mask = build_shifted_window_mask(
                height,
                width,
                self.window_size,
                self.shift_size,
                x.device,
            )
            expanded_mask = base_mask.repeat(batch, 1, 1)
            scores = scores.masked_fill(
                ~expanded_mask[:, None],
                torch.finfo(scores.dtype).min,
            )

        weights = scores.softmax(dim=-1)
        attended = weights @ v
        attended = attended.transpose(1, 2).contiguous().flatten(2)
        attended = self.out(attended)

        x = window_reverse(
            attended,
            self.window_size,
            height,
            width,
            batch,
        )

        if self.shift_size > 0:
            x = torch.roll(
                x,
                shifts=(self.shift_size, self.shift_size),
                dims=(1, 2),
            )

        return residual + x


feature = torch.randn(2, 4, 4, 24, device=device)
w_msa = SwinWindowAttention(shift_size=0).to(device)
sw_msa = SwinWindowAttention(shift_size=1).to(device)

feature = w_msa(feature)
feature = sw_msa(feature)
print("two-block Swin output:", feature.shape)


## 5. Patch merging


In [ ]:
x = feature
merged = torch.cat(
    [
        x[:, 0::2, 0::2],
        x[:, 1::2, 0::2],
        x[:, 0::2, 1::2],
        x[:, 1::2, 1::2],
    ],
    dim=-1,
)
merge_norm = nn.LayerNorm(4 * 24).to(device)
merge_reduction = nn.Linear(4 * 24, 2 * 24, bias=False).to(device)
merged = merge_reduction(merge_norm(merged))
print("patch merged:", merged.shape)


## 6. FPN


In [ ]:
class TinyFPN(nn.Module):
    def __init__(self, channels=(32, 64, 128), out_channels=24):
        super().__init__()
        self.lateral = nn.ModuleList(
            [nn.Conv2d(c, out_channels, 1) for c in channels]
        )
        self.smooth = nn.ModuleList(
            [
                nn.Conv2d(out_channels, out_channels, 3, padding=1)
                for _ in channels
            ]
        )

    def forward(self, features):
        c3, c4, c5 = features
        p5_inner = self.lateral[2](c5)
        p4_inner = self.lateral[1](c4) + F.interpolate(
            p5_inner,
            size=c4.shape[-2:],
            mode="nearest",
        )
        p3_inner = self.lateral[0](c3) + F.interpolate(
            p4_inner,
            size=c3.shape[-2:],
            mode="nearest",
        )

        return (
            self.smooth[0](p3_inner),
            self.smooth[1](p4_inner),
            self.smooth[2](p5_inner),
        )


fpn = TinyFPN().to(device)
p3, p4, p5 = fpn(
    (
        torch.randn(2, 32, 16, 16, device=device),
        torch.randn(2, 64, 8, 8, device=device),
        torch.randn(2, 128, 4, 4, device=device),
    )
)
print("FPN:", p3.shape, p4.shape, p5.shape)


## 7. CenterNet-style heads -> decode


In [ ]:
class CenterNetHead(nn.Module):
    def __init__(self, channels=24, classes=3):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(),
        )
        self.heatmap = nn.Conv2d(channels, classes, 1)
        self.offset = nn.Conv2d(channels, 2, 1)
        self.size = nn.Conv2d(channels, 2, 1)

    def forward(self, x):
        h = self.shared(x)
        return {
            "heatmap": self.heatmap(h).sigmoid(),
            "offset": self.offset(h),
            "size": F.softplus(self.size(h)),
        }


def decode_centernet(prediction):
    heatmap = prediction["heatmap"]
    batch, classes, height, width = heatmap.shape

    score, flat_index = heatmap.view(batch, -1).max(dim=-1)

    class_id = flat_index // (height * width)
    spatial = flat_index % (height * width)
    y = spatial // width
    x = spatial % width

    batch_id = torch.arange(batch, device=heatmap.device)
    offset = prediction["offset"][batch_id, :, y, x]
    size = prediction["size"][batch_id, :, y, x]

    center_x = x.float() + offset[:, 0]
    center_y = y.float() + offset[:, 1]

    x1 = center_x - 0.5 * size[:, 0]
    y1 = center_y - 0.5 * size[:, 1]
    x2 = center_x + 0.5 * size[:, 0]
    y2 = center_y + 0.5 * size[:, 1]

    boxes = torch.stack([x1, y1, x2, y2], dim=-1)
    return score, class_id, boxes


head = CenterNetHead().to(device)
prediction = head(p3)
score, class_id, boxes = decode_centernet(prediction)

loss = (
    prediction["heatmap"].mean()
    + prediction["offset"].square().mean()
    + prediction["size"].mean()
)
loss.backward()

print("boxes:", boxes)
print("head grad:", head.shared[0].weight.grad.norm().item())


## References and provenance

- **Swin Transformer**: window attention, relative-position bias, cyclic shift,
  and the shifted-window attention mask are all executed.
- **FPN**: lateral 1x1 projections, top-down upsampling, and smoothing.
- **CenterNet**: prediction heads feed the actual center/offset/size decode path.
